# Backtesting Engine — Example Usage

Demonstrates the Phase 2 backtesting framework:
- `BaseStrategy` — abstract strategy interface (Backtrader-compatible)
- `BacktestEngine` — high-level wrapper around Backtrader Cerebro
- `BaseSignal` — signal generation interface
- Risk metrics — Sharpe, Sortino, max drawdown, etc.
- Performance reporting — equity curve, trade log

**Example strategy:** Simple moving average crossover (illustrative only; not investment advice).

> **Note:** This notebook uses **synthetic data** for demonstration.  
> **Clear all outputs before committing** (`Cell → All Output → Clear`).

In [ ]:
import sys
from pathlib import Path

# Make the package importable
repo_root = Path().resolve().parents[1]
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Repo root: {repo_root}")

## 1. Generate Synthetic Price Data

Create a simple synthetic OHLCV DataFrame for testing.  
Real strategies would use the data pipeline from Phase 1.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Synthetic price walk with trend
n = 252
base_price = 100.0
returns = np.random.randn(n) * 0.015 + 0.0003  # slight upward drift
prices = base_price * (1 + returns).cumprod()

# Build OHLCV
dates = pd.date_range("2024-01-01", periods=n, freq="D", tz="UTC")
df = pd.DataFrame(
    {
        "open": prices * (1 + np.random.randn(n) * 0.002),
        "high": prices * (1 + np.abs(np.random.randn(n)) * 0.005),
        "low": prices * (1 - np.abs(np.random.randn(n)) * 0.005),
        "close": prices,
        "volume": np.random.randint(100_000, 1_000_000, n),
    },
    index=dates,
)

print(f"Synthetic data: {len(df)} rows")
df.head()

## 2. Define a Simple Strategy — MA Crossover

Subclass `BaseStrategy` and implement `next()`.  
This is a **trivial illustrative strategy** — not production-ready.

In [ ]:
import backtrader as bt
from quant_trading.backtesting.base_strategy import BaseStrategy


class SimpleMAStrategy(BaseStrategy):
    """Moving average crossover strategy.
    
    Buy when fast MA crosses above slow MA.
    Sell when fast MA crosses below slow MA.
    """
    params = (
        ("fast_period", 10),
        ("slow_period", 50),
    )

    def __init__(self):
        self.ma_fast = bt.indicators.SMA(self.data.close, period=self.p.fast_period)
        self.ma_slow = bt.indicators.SMA(self.data.close, period=self.p.slow_period)
        self.crossover = bt.indicators.CrossOver(self.ma_fast, self.ma_slow)

    def next(self):
        if self.crossover > 0:  # fast crossed above slow
            if not self.position:
                self.buy()
        elif self.crossover < 0:  # fast crossed below slow
            if self.position:
                self.close()


print("SimpleMAStrategy defined.")

## 3. Run the Backtest

Use `BacktestEngine` to wire up data, strategy, and commission model.

In [ ]:
from quant_trading.backtesting.engine import BacktestEngine
from quant_trading.backtesting.sizing import FixedFractionalSizer

engine = BacktestEngine(
    initial_cash=100_000,
    commission=0.001,  # 0.1% per trade
    slippage_perc=0.0,
)

# Allocate 10% of portfolio value per trade instead of Backtrader's default (1 share)
engine.set_sizer(FixedFractionalSizer, fraction=0.10)

engine.add_data(df, name="SYNTHETIC")
engine.add_strategy(SimpleMAStrategy, fast_period=10, slow_period=50)

results = engine.run()

print(f"Initial value : ${results['initial_value']:,.2f}")
print(f"Final value   : ${results['final_value']:,.2f}")
print(f"Total return  : {(results['final_value'] / results['initial_value'] - 1) * 100:.2f}%")

### Comparing sizers

Run the same strategy three ways to see how sizing affects outcomes:
- **Default** — Backtrader's built-in sizer (1 share per order)
- **FixedFractionalSizer** — 10% of portfolio per trade
- **VolatilityTargetedSizer** — size calibrated to 15% annualized vol target

In [ ]:
import matplotlib.pyplot as plt
from quant_trading.backtesting.sizing import FixedFractionalSizer, VolatilityTargetedSizer


def run_with_sizer(sizer_class=None, **sizer_kwargs):
    """Helper: build a fresh engine, optionally apply a sizer, and run."""
    eng = BacktestEngine(initial_cash=100_000, commission=0.001)
    if sizer_class is not None:
        eng.set_sizer(sizer_class, **sizer_kwargs)
    eng.add_data(df, name="SYNTHETIC")
    eng.add_strategy(SimpleMAStrategy, fast_period=10, slow_period=50)
    return eng.run()


results_default = run_with_sizer()                                               # 1 share per trade
results_fixed   = run_with_sizer(FixedFractionalSizer, fraction=0.10)           # 10% per trade
results_vol     = run_with_sizer(VolatilityTargetedSizer, target_vol=0.15, lookback=20)  # vol-targeted

fig, ax = plt.subplots(figsize=(12, 4))
results_default["equity_curve"]["value"].plot(ax=ax, label="Default (1 share)")
results_fixed["equity_curve"]["value"].plot(ax=ax, label="FixedFractional 10%")
results_vol["equity_curve"]["value"].plot(ax=ax, label="VolatilityTargeted 15%")
ax.axhline(100_000, color="grey", linestyle="--", linewidth=0.8, label="Initial")
ax.set_title("Equity Curve — Sizer Comparison")
ax.set_ylabel("Portfolio Value (USD)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary table
import pandas as pd
summary = pd.DataFrame(
    {
        "Sizer": ["Default (1 share)", "FixedFractional 10%", "VolatilityTargeted 15%"],
        "Final Value": [
            results_default["final_value"],
            results_fixed["final_value"],
            results_vol["final_value"],
        ],
        "Sharpe": [
            results_default["metrics"]["sharpe"],
            results_fixed["metrics"]["sharpe"],
            results_vol["metrics"]["sharpe"],
        ],
        "Max DD %": [
            results_default["metrics"]["max_dd"] * 100,
            results_fixed["metrics"]["max_dd"] * 100,
            results_vol["metrics"]["max_dd"] * 100,
        ],
    }
).set_index("Sizer").round(4)
summary

## 4. Performance Metrics

Examine the computed risk metrics: Sharpe, Sortino, max drawdown, etc.

In [ ]:
metrics = results["metrics"]

print("Performance Metrics:")
print(f"  Ann. Return      : {metrics['ann_return'] * 100:.2f}%")
print(f"  Ann. Volatility  : {metrics['ann_volatility'] * 100:.2f}%")
print(f"  Sharpe Ratio     : {metrics['sharpe']:.2f}")
print(f"  Sortino Ratio    : {metrics['sortino']:.2f}")
print(f"  Max Drawdown     : {metrics['max_dd'] * 100:.2f}%")
print(f"  Max DD Duration  : {metrics['max_dd_duration_days']} days")

## 5. Equity Curve

Plot the portfolio value over time.

In [ ]:
import matplotlib.pyplot as plt

equity = results["equity_curve"]["value"]

fig, ax = plt.subplots(figsize=(11, 4))
equity.plot(ax=ax, title="Equity Curve — SimpleMAStrategy", ylabel="Portfolio Value (USD)", color="steelblue")
ax.axhline(results["initial_value"], color="grey", linestyle="--", linewidth=0.8, label="Initial")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Trade Log

Inspect individual trades — open/close dates and PnL.

In [ ]:
trades = results["trades"]

if not trades.empty:
    print(f"{len(trades)} trades executed:")
    print(trades.head(10))
else:
    print("No trades executed.")

## 7. Using BaseSignal — Example

`BaseSignal` provides a cleaner interface for signal generation outside of Backtrader.  
Below is a simple MA crossover signal (same logic as the strategy).

In [ ]:
from quant_trading.signals.base import BaseSignal


class MASignal(BaseSignal):
    """Moving average crossover signal."""
    def __init__(self, fast=10, slow=50):
        self.fast = fast
        self.slow = slow

    def generate(self, prices: pd.DataFrame, features: pd.DataFrame | None = None) -> pd.Series:
        self.validate_prices(prices, min_rows=max(self.fast, self.slow))
        close = prices["close"]
        ma_fast = close.rolling(self.fast).mean()
        ma_slow = close.rolling(self.slow).mean()
        signal = pd.Series(0, index=prices.index)
        signal[ma_fast > ma_slow] = 1
        signal[ma_fast < ma_slow] = -1
        return signal


# Generate signal
signal_gen = MASignal(fast=10, slow=50)
signals = signal_gen.generate(df)

print(f"Signal distribution:\n{signals.value_counts()}")
signals.tail(20)